# Consumo ilimitado: o link que vazou num fórum

OWASP Top 10 para LLMs 2025 — **LLM10 Unbounded Consumption**

Modelo real no Ollama (`campus/`), local e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, llm  # noqa: E402

TENTATIVAS = 6  # um modelo real sorteia: cada ataque é medido em várias sementes
print("Modelo real:", llm.MODELO, "no Ollama", llm.versao(), "- os números variam entre máquinas. Uso educacional.")

Modelo real: llama3.2:3b no Ollama 0.40.14 - os números variam entre máquinas. Uso educacional.


## Cenário
Terça-feira: cerca de 400 chamadas de alunos ao Aurora Assist. Quinta-feira, um fórum externo compartilha o endpoint e poucos endereços passam a usá-lo como tradutor gratuito: cerca de 38 mil chamadas no dia. Tokens, teto e limite são medidos em chamadas reais; o volume do dia e o custo (US$ 0,029 por chamada) são simulados.

In [2]:
CUSTO_POR_CHAMADA = 0.029  # dólares, valor simulado
LIMITE_POR_ORIGEM = 50      # chamadas por origem por dia
PEDIDO = [{"role": "user", "content": "Traduza para o inglês e explique em detalhes, com 20 exemplos, o que é segurança de modelos de linguagem."}]

def dia(terceiros):
    """200 alunos (2 chamadas cada) + `terceiros` chamadas vindas de 5 endereços externos."""
    alunos = ["aluno-{:03d}".format(i % 200) for i in range(400)]
    externos = ["externo-{}".format(i % 5) for i in range(terceiros)]
    return alunos + externos

def simular(chamadas, limitador=None):
    atendidas = [o for o in chamadas if limitador is None or limitador.permitir(o)]
    alunos_bloqueados = sum(1 for o in chamadas if o.startswith("aluno")) - \
        sum(1 for o in atendidas if o.startswith("aluno"))
    return len(atendidas) * CUSTO_POR_CHAMADA, len(chamadas) - len(atendidas), alunos_bloqueados

terca, quinta = dia(0), dia(37600)


## Sem defesa

In [3]:
# Uma chamada real: quanto um pedido de terceiro consome, medido pelo Ollama.
sem_teto = llm.LLM(max_tokens=2048)
sem_teto.gerar(PEDIDO)
print("uma chamada sem teto de tokens:", sem_teto.tokens_entrada, "de entrada +", sem_teto.tokens_saida, "gerados")

# Escala do cenário: o volume é simulado; rodar 38 mil chamadas aqui levaria dias.
for nome, chamadas in (("terça", terca), ("quinta", quinta)):
    custo, _, _ = simular(chamadas)
    print("{:<7} {:>6} chamadas  US$ {:>8.2f}".format(nome, len(chamadas), custo))


uma chamada sem teto de tokens: 56 de entrada + 1136 gerados
terça      400 chamadas  US$    11.60
quinta   38000 chamadas  US$  1102.00


## Com defesa
`defesas.LimitadorTaxa` limita as chamadas **por origem**. Cada aluno é uma origem própria, então o uso legítimo não é afetado.

In [4]:
# Teto de tokens: mesma chamada real, com limite por resposta.
com_teto = llm.LLM(max_tokens=128)
com_teto.gerar(PEDIDO)
print("com teto de tokens:", com_teto.tokens_saida, "gerados, contra", sem_teto.tokens_saida)
assert com_teto.tokens_saida <= 128

# Limite de taxa: uma rajada real de 10 chamadas da mesma origem, com limite de 3.
limitador = defesas.LimitadorTaxa(3)
modelo = llm.LLM(max_tokens=16)
for i in range(10):
    if limitador.permitir("externo-0"):
        modelo.gerar([{"role": "user", "content": "Traduza para o inglês a frase {}: bom dia.".format(i)}])
print("chamadas que chegaram ao modelo:", modelo.chamadas, "de 10")
assert modelo.chamadas == 3

custo, bloqueadas, alunos_bloqueados = simular(quinta, defesas.LimitadorTaxa(LIMITE_POR_ORIGEM))
print("quinta com limite: US$ {:.2f}  ({} chamadas bloqueadas, {} delas de alunos)".format(
    custo, bloqueadas, alunos_bloqueados))
assert custo < 20 and alunos_bloqueados == 0


com teto de tokens: 128 gerados, contra 1136


chamadas que chegaram ao modelo: 3 de 10
quinta com limite: US$ 18.85  (37350 chamadas bloqueadas, 0 delas de alunos)


## O que observar
- Nenhuma chamada isolada parecia maliciosa: o problema é o volume vindo de poucas origens.
- "Denial of wallet": o serviço continua no ar e a conta explode; sem alerta, só se percebe na fatura.
- O limite precisa ser por usuário/origem. Um limite único para "alunos" bloquearia o uso legítimo junto com o abuso.
- Limite de taxa é o controle mais barato de ligar no dia 1; quota por usuário, teto de tokens, autenticação do endpoint e alerta de custo completam a defesa.